<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🔍 M3 作业 · 自动 Code Review Flow

**一句话**：把 M1/M2 的代码审查 crew 放进 Flow：先判断 PR 改动大不大，小改动一次 LLM 调用就审完，大改动才让 crew（质量与安全并行）深度审查。

<span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🏷️ 作业 Homework</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">📄 C1M3_Assignment.ipynb</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">▶️ cd code_review_flow &amp;&amp; crewai run</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🧩 前置：M2 作业 code review crew · M3 Lab 2 Flow</span>

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**要解决的问题**：改个缩进、修个 typo 也让三个 agent 跑一遍，既慢又贵。
本作业：
- **Flow** 负责分流：读 PR diff → LLM 判断 SIMPLE / COMPLEX → 简单审查 或 crew 深度审查 → 最终决定；
- **Crew** 内部提速：代码质量和安全审查**并行**，原来的 Review Decision 换成 **Summarize Findings**（输出结构化 JSON）；
- state 记录 token 用量，最后算一次运行花了多少钱。

所有要填的都在 `code_review_flow/` 项目文件里，notebook 只负责画图、运行和算成本。

</div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

📝 **项目文件做过的修复（2026-10-21）**
1. **所有 `.py` 和 README 原本是 notebook JSON**（下载课程时被转成了 `{"nbformat": 4, ...}`），Python 根本 import 不了 → 已还原成普通源码（按 cell 顺序拼接，内容不变）。
2. **10 个练习的 `None` 空位原本都没填** → 已按题目要求填好（本作业没有 `Solution/` 文件夹，是自己写的答案，下面逐个讲）。
3. `crew.py` 的 `SerperDevTool(base_url=os.getenv("DLAI_SERPER_BASE_URL"))`：课程平台有这个代理变量，本地没有 → `None` → pydantic 报 `base_url Input should be a valid string`。改成 `os.getenv("DLAI_SERPER_BASE_URL", "https://google.serper.dev")`。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🗂️ 1. 项目结构

</div>

<img src="images/folder-structure.png" width="200">

<div style="width:97%;"><div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(230px,1fr)); gap:10px;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #a78bfa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">👥 crews/code_review_crew/</div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">config/agents.yaml</code>（不用改）、<code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">config/tasks.yaml</code>（练习 1）、<code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">crew.py</code>（练习 2）、<code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">guardrails/guardrails.py</code>（安全审查输出检查）。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #4ade80; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🌊 main.py</div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">ReviewState</code> + <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">PRCodeReviewFlow</code> + <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">kickoff()</code> / <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">plot()</code>（练习 3–9）。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">📄 pyproject.toml</div><div style="font-size:0.92em; line-height:1.6;">钉死 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">crewai[tools]==0.203.1</code> + <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">dill</code>；<code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">crewai run</code> → <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">uv run kickoff</code>。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #fbbf24; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">📁 ../files/</div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">code_changes.txt</code>（改了登录逻辑，含 SQL 注入）和 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">code_changes_simple.txt</code>（只改了 import 别名）。</div></div></div></div>

In [1]:
import os, sys
from pathlib import Path

# VSCode 的 kernel PATH 不一定含 .venv/bin → 放到最前，`!crewai` / `uv` 才找得到
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"
os.environ.pop("VIRTUAL_ENV", None)   # 项目用自己的 .venv（crewai 0.203.1），免得 uv 报 VIRTUAL_ENV 不匹配警告

PROJ = Path("code_review_flow")
SRC = PROJ / "src" / "code_review_flow"
CREW_DIR = SRC / "crews" / "code_review_crew"
MAIN = (SRC / "main.py").read_text()

def between(text, start, end=None):
    """打印 text 中从 start 所在行到 end 所在行之前的片段"""
    a = text.rfind("\n", 0, text.index(start)) + 1
    b = text.rfind("\n", 0, text.index(end, a)) + 1 if end else len(text)
    print(text[a:b].rstrip())

print((PROJ / "pyproject.toml").read_text())

[project]
name = "code_review_flow"
version = "0.1.0"
description = "code_review_flow using crewAI"
authors = [{ name = "Your Name", email = "you@example.com" }]
requires-python = ">=3.10,<3.14"
dependencies = [
    "crewai[tools]==0.203.1",
    "dill"
]

[project.scripts]
kickoff = "code_review_flow.main:kickoff"
run_crew = "code_review_flow.main:kickoff"
plot = "code_review_flow.main:plot"

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[tool.crewai]
type = "flow"
project_id = "3623fe43-9131-4483-baa6-67608f9b5933"



<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧠 2. Crew：两个审查并行，再汇总

</div>

<img src="./images/M2agents-tasks-diagram.png" width="520">  <img src="./images/parallel-agents-tasks-diagram.png" width="520">

<div style="background:#111827; border:1px dashed #334155; border-radius:8px; padding:10px 16px; width:97%;"><div style="color:#94a3b8; font-weight:700; margin-bottom:6px;">前两个同时启动；summarize 是同步 task，开始前先等它们都完成</div><span style="display:inline-block; background:#0f172a; border:1.5px solid #fbbf24; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">analyze_code_quality</code> ⚡async（Senior Developer）</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #fbbf24; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">review_security</code> ⚡async（Security Engineer + Serper/Scrape，guardrail）</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #4ade80; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">summarize_findings</code>（Tech Lead，context = 两者）</span></div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 练习 1：tasks.yaml —— summarize_findings 的输出与 context

</div>

- `expected_output`：写明 JSON 的 4 个 key 和类型：`confidence`（0–100 **整数**）、`findings`（**字符串**）、`fix`（**列表**，每项含问题、解决办法、原因）、`recommendations`。
- `context`：`[analyze_code_quality, review_security]`。写全两个，才能保证**等两者都结束**并拿到它们的结果。

In [2]:
tasks_yaml = (CREW_DIR / "config/tasks.yaml").read_text()
between(tasks_yaml, "summarize_findings:")

summarize_findings:
  description: >
        1. Review the findings from both the code quality and security analyses.
        2. Determine if the pull request can be automatically approved based on these findings.
        3. If approval is not possible, decide whether to request specific changes from the developer or escalate to human review.
        4. Provide a clear explanation of your decision and any required next steps.
        PR content: \n{file_content}
  ### START CODE HERE ###
  # write the expected output in detail following the exercise instructions
  expected_output: >
    A JSON with keys:
    'confidence' (integer between 0 and 100, your confidence score of the code changes),
    'findings' (string, a summary of the key findings from both the code quality and security analyses),
    'fix' (a list of what must be fixed, each item with the description of the issue, the possible solutions
    and the explanation of why the fix is needed), and
    'recommendations' (any add

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 练习 2：crew.py —— 装饰器、async、输出模型

</div>

三件事：
1. 补齐 `@agent`（security_engineer、tech_lead）、`@task`（review_security、summarize_findings）、`@crew`：**没加装饰器的方法不会被收集**进 `self.agents` / `self.tasks`，yaml 里按名字引用时就找不到；
2. `analyze_code_quality` 和 `review_security` 设 `async_execution=True`；
3. `SummarizedFindingsJSON`：`confidence: int`、`findings: str`、`fix: list[Fix]`、`recommendations: list[str]`。

In [3]:
crew_py = (CREW_DIR / "crew.py").read_text()
between(crew_py, "@CrewBase")

@CrewBase
class CodeReviewCrew:
    """CodeReview crew"""
    @agent
    def senior_developer(self) -> Agent:       
        return Agent(
            config=self.agents_config["senior_developer"],
            llm="gpt-4o-mini",
        )

    ### START CODE HERE ###
    @agent
    ### END CODE HERE ###
    def security_engineer(self) -> Agent:
        os.environ["SERPER_API_KEY"] = get_serper_api_key()
        return Agent(
            config=self.agents_config["security_engineer"],
            tools=[
                # add the SerperDevTool to the agent's tools
                SerperDevTool(search_url="https://owasp.org",
                              base_url=os.getenv("DLAI_SERPER_BASE_URL", "https://google.serper.dev")),  # 课程平台用代理地址；本地没设这个变量 → 用官方地址
                ScrapeWebsiteTool() 
                ],
            llm="gpt-4o-mini"
        )
    
    ### START CODE HERE ###
    @agent
    ### END CODE HERE ###
    def tech_lead(self) -> Agent:
        return Agent(
            

<div style="width:97%;"><div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(230px,1fr)); gap:10px;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🧾 output_json=Model</div><div style="font-size:0.92em; line-height:1.6;">让 task 输出符合 pydantic 模型的 JSON；结果在 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">TaskOutput.json_dict</code> / <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">CrewOutput.json_dict</code>。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #f87171; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🛡️ guardrails=[…]</div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">security_review_output_guardrail</code> 检查风险等级只能是 low/medium/high，且 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">highest_risk</code> 与各漏洞等级一致；不通过就让 agent 重做。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #a78bfa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🧠 memory=True</div><div style="font-size:0.92em; line-height:1.6;">crew 级短期/长期/实体记忆（日志里的 Memory Retrieval）。</div></div></div></div>

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**题目说 `recommendations` 是 string，模板却是 `list[None]`** —— 按模板写成 `list[str]`。`output_json` 以 pydantic 模型为准，yaml 里的文字描述只是给 LLM 的提示，两者冲突时以模型为准，所以最好让两处一致。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧠 3. Flow：分流 → 审查 → 决定

</div>

<img src="images/flow-diagram.png" width="300">

<div style="background:#111827; border:1px dashed #334155; border-radius:8px; padding:10px 16px; width:97%;"><div style="color:#94a3b8; font-weight:700; margin-bottom:6px;">正常路径</div><span style="display:inline-block; background:#0f172a; border:1.5px solid #a78bfa; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">read_pr_file</code> @start</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #fbbf24; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">analyze_changes</code> @router</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #60a5fa; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">"SIMPLE" → <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">simple_review</code>（1 次 LLM）</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #f87171; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">"COMPLEX" → <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">full_crew_review</code>（crew）</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #4ade80; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">make_final_decision</code> = or_(两者)</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #4ade80; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">return_final_answer</code></span></div>

<div style="background:#111827; border:1px dashed #334155; border-radius:8px; padding:10px 16px; width:97%;"><div style="color:#94a3b8; font-weight:700; margin-bottom:6px;">出错路径：<code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">@listen(or_(&quot;ERROR&quot;, &quot;make_final_decision&quot;))</code></div><span style="display:inline-block; background:#0f172a; border:1.5px solid #f87171; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">read_pr_file</code> 读文件失败 → errors 非空</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #fbbf24; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">router 返回 "ERROR"</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #4ade80; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">return_final_answer</code> 直接输出错误</span></div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 练习 3：state

</div>

每个字段写**类型 + 初始值**。`tokens_used` 存 crew 的 token 统计，后面算成本用。

In [4]:
between(MAIN, "# define the flow state", "@persist()")

# define the flow state
class ReviewState(BaseModel):
    # already dive the path to the PR diff file
    pr_file_path: str = "../files/code_changes.txt"
    ### START CODE HERE ###
    pr_content: str = "" # initialize with an empty string
    errors: list = [] # initialize with an empty list
    review_result: dict = {} # set the correct type
    crew_needed: bool = False # set the correct type
    tokens_used: dict = {} # initialize with an empty dict
    final_answer: str = "" # initialize with an empty string
    ### END CODE HERE ###


<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 练习 4：`read_pr_file`（@start）

</div>

入口用 `@start()`；读成功 → `self.state.pr_content = file_contents`；失败 → 往 `errors` 追加信息，并把 `error_message` 写进 `final_answer`。

In [5]:
between(MAIN, "@persist()", "### START CODE HERE ###\n    @router")

@persist()
# Define the flow
class PRCodeReviewFlow(Flow[ReviewState]):
    """
    CrewAI Flow for automated code review of pull requests.
    """
    ### START CODE HERE ###
    @start()
    ### END CODE HERE ###
    def read_pr_file(self):
        """Read the PR file and determine if crew review is needed"""
        print("🔍 Starting PR Code Review...")
        
        # Get the file path from the state
        pr_file_path = self.state.pr_file_path

        if not pr_file_path:
            # append some error message to the errors state variable
            self.state.errors.append("Missing 'file_path' in state") 
            # save the error message in the final answer state variable
            self.state.final_answer = f"There was no file_path set. Please set the 'file_path' in the state and try again." 
            print(f"❌ {self.state.final_answer}")
            return

        # try reading the file
        try:
            with open(pr_file_path, "r") as f:
               

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 练习 5：`analyze_changes`（@router）

</div>

`@router("read_pr_file")`。有错误 → 返回 `"ERROR"`；否则让 LLM 回一个词，`COMPLEX` → `crew_needed=True` 返回 `"COMPLEX"`，否则 `crew_needed=False` 返回 `"SIMPLE"`。
签名里的 `context` 参数会收到上游方法 `read_pr_file` 的返回值（这里是 `None`）。

In [6]:
between(MAIN, "### START CODE HERE ###\n    @router", "# if the PR is simple")

    ### START CODE HERE ###
    @router("read_pr_file")
    ### END CODE HERE ###
    def analyze_changes(self, context):
        """Route to appropriate review type based on complexity"""

        ### START CODE HERE ###
        # if there are any errors in the state variable, return 'ERROR'
        if len(self.state.errors) > 0:
            return "ERROR"
        ### END CODE HERE ###

        else:
            # define the prompt to analyze the changes
            prompt = (
                "Analyze this pull request diff file and respond with exactly one word: SIMPLE or COMPLEX.\n"
                "SIMPLE: small changes that don't compromise code quality or security. For example typos, minor refactoring, formatting, small doc changes"
                "COMPLEX: bigger changes that need closer inspection. For example new features, bug fixes, logic changes, security-relevant changes, or anything requiring deep review or best practices research.\n"
                f"\nPR Diff:\n{self.s

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 练习 6：`simple_review`（SIMPLE 分支）

</div>

`@listen("SIMPLE")`；prompt 末尾补上输出格式（confidence / findings / recommendations）和 **PR 内容**（`self.state.pr_content`），结果存进 `review_result`。

In [7]:
between(MAIN, "# if the PR is simple", "# if the PR is complex")

    # if the PR is simple, do a simple review
    ### START CODE HERE ###
    @listen("SIMPLE")
    ### END CODE HERE ###
    def simple_review(self):
        """Simple review for minor changes"""
        print("⚡ Performing simple review...")
        
        ### START CODE HERE ###
        # Complete the last part of the prompt
        prompt = (
            "Analyze this pull request diff file and evaluate the changes.\n"
            "Do not make assumptions or considerations about the code outside of the diff provided, but if warranted you can make suggestions.\n"
            "Return a JSON dictionary with the following keys:\n"
            "- confidence: your confidence score (integer between 0 and 100) of the code changes\n"
            "- findings: a summary of the key findings of your analysis\n"
            "- recommendations: any additional recommendations or observations\n"
            f"\nPR Diff:\n{self.state.pr_content}\n")
        ### END CODE HERE ###

        # define 

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**类型细节**：`llm.call()` 返回的是**字符串**（常带 ```json 围栏），而 `review_result` 声明为 `dict`。模板只给一行，所以这里写成 `{"simple_review": result}`：类型对得上，下一步 prompt 照样能读到全文。
（直接 `= result` 也能跑：pydantic 默认不在赋值时校验，但 state 里就混进了 str。）

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 练习 7：`full_crew_review`（COMPLEX 分支）

</div>

`@listen("COMPLEX")`；`code_review_crew.kickoff(inputs={'file_content': pr_content})`——key 必须是 tasks.yaml 里的占位符 `{file_content}`。
存 `result.json_dict`（最后一个 task 的 `output_json`）和 `result.token_usage.model_dump()`。

In [8]:
between(MAIN, "# if the PR is complex", "# make the final decision")

    # if the PR is complex, deploy crew review
    ### START CODE HERE ###
    @listen("COMPLEX")
    ### END CODE HERE ###
    def full_crew_review(self):
        """Full crew review for complex changes"""
        print("🚀 Starting full crew review...")
        
        # get the PR content from the state variable
        pr_content = self.state.pr_content

        # create the crew
        code_review_crew = CodeReviewCrew().crew()

        try: 
            ### START CODE HERE ###
            # kickoff the crew. Pass the PR content in the inputs
            result = code_review_crew.kickoff(inputs={'file_content': pr_content})

            # save the results in the state variable. You can use the `json_dict` attribute of the result
            self.state.review_result = result.json_dict

            # save the tokens used by the crew. You can use the `token_usage` attribute of the result
            self.state.tokens_used = result.token_usage.model_dump()
            ### END CODE HE

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**为什么 `.model_dump()`？** `token_usage` 是 pydantic 的 `UsageMetrics` 对象，而 state 字段是 `dict`，并且 `@persist` / `json.dump(state.model_dump())` 都要把 state 序列化。先转成 dict，`flow_state.json` 里才会有 `prompt_tokens` 这些 key。
**注意**：`CodeReviewCrew().crew()` 写在 `try` **外面**，建 crew 时出错（比如上面的 `base_url=None`）不会进 except，整个 flow 直接崩。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 练习 8–9：`make_final_decision` 与 `kickoff()`

</div>

`@listen(or_("simple_review", "full_crew_review"))`：两条分支只会走一条，任一完成就触发；`llm.call(messages=prompt)` 的结果存 `final_answer`。`kickoff()` 里 `PRCodeReviewFlow(tracing=True)` 再 `.kickoff(inputs={"id": "my_code_review_flow"})`，然后把结果和 state 存成 `../result.dill`、`../flow_state.json`（供评分）。

In [9]:
between(MAIN, "# make the final decision")

    # make the final decision based on the review results
    ### START CODE HERE ###
    @listen(or_("simple_review", "full_crew_review"))
    ### END CODE HERE ###
    def make_final_decision(self):
        """Make the final decision based on the review results"""
        print("🧐 Making final decision based on review results...")
        
        # get the review result from the state variable
        review_result = self.state.review_result

        # use an LLM call to make the final decision and generate the final message
        prompt = (
            "Based on the following analysis of the pull request diff file, "
            "make a final decision on whether to approve the PR for merging.\n" \
            "Any review with a confidence score above 85 can be approves, but improvements can be suggested.\n"
            "Return a full report with:\n"
            "- Final Review Decision: APPROVE (the PR is good to be approved), REQUEST CHANGES (the PR needs some modifications, but

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🗺️ 4. 画出 Flow

</div>

第一次运行会先按 pyproject 建项目 `.venv`（crewai 0.203.1）。

In [10]:
! cd code_review_flow && crewai flow plot

Plotting the Flow


╭─────────────────────────────── Flow Execution ───────────────────────────────╮
│                                                                              │
│  Starting Flow Execution                                                     │
│  Name: PRCodeReviewFlow                                                      │
│  ID: 982736f7-095b-401a-9cdc-13558f7d65ef                                    │
│  Tool Args:                                                                  │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

Plot saved as crewai_flow.html


0.203.1 的 `plot()` 把 `crewai_flow.html` 直接写在 `code_review_flow/` 里（和课程说的一样；Lab 2 用的 1.3.0 改成了写临时目录并自动开浏览器）。在 VSCode 里右键该文件 → 用浏览器打开。

In [11]:
print((PROJ / "crewai_flow.html").exists(), (PROJ / "crewai_flow.html").stat().st_size, "bytes")

True 45234 bytes


<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🚀 5. 练习 10：运行 Flow

</div>

`crewai run` → `uv run kickoff` → `main.kickoff()`。默认审查 `../files/code_changes.txt`（复杂改动）→ 走 crew，约 1 分钟。
开头若打印 `Loading flow state from memory for UUID: my_code_review_flow`，说明 `@persist` 恢复了上一次的 state。

In [12]:
! cd code_review_flow && crewai run

╭─────────────────────────────── Flow Execution ───────────────────────────────╮
│                                                                              │
│  Starting Flow Execution                                                     │
│  Name: PRCodeReviewFlow                                                      │
│  ID: 1be05d4c-f28e-4994-9edf-92269d2dc4f9                                    │
│  Tool Args:                                                                  │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

Loading flow state from memory for UUID: my_code_review_flow


🌊 Flow: PRCodeReviewFlow
ID: my_code_review_flow
└── 🧠 Starting Flow...

Flow started with ID: my_code_review_flow
🌊 Flow: PRCodeReviewFlow
ID: my_code_review_flow
🌊 Flow: PRCodeReviewFlow
ID: my_code_review_flow
├── 🧠 Starting Flow...
🔍 Starting PR Code Review...
🌊 Flow: PRCodeReviewFlow
ID: my_code_review_flow
├── 🧠 Starting Flow...
🌊 Flow: PRCodeReviewFlow
ID: my_code_review_flow
├── Flow Method Step
🌊 Flow: PRCodeReviewFlow
ID: my_code_review_flow
├── Flow Method Step
├── ✅ Completed: read_pr_file
└── 🔄 Running: analyze_changes

🌊 Flow: PRCodeReviewFlow
ID: my_code_review_flow
├── Flow Method Step
├── ✅ Completed: read_pr_file
└── 🔄 Running: analyze_changes

🌊 Flow: PRCodeReviewFlow
ID: my_code_review_flow
├── Flow Method Step
├── ✅ Completed: read_pr_file
└── 🔄 Running: analyze_changes

...

✅ Crew: crew
├── 📋 Task: Analyze Code Quality (ID: 952ee0e6-91ed-4f4c-9946-3e2750461c4e)
│   Assigned to: Senior Developer
│   
│   Status: ✅ Completed
├── 📋 Task: Review Security (ID: 417079a1-1992-46d3-9238-00e9c7d98508)
│   Assigned to: Security Engineer
│   
│   Status: ✅ Completed
│   ├── 🔧 Used Search the internet with Serper (1)
│   ├── 🧠 Memory Update Overall
│   │   ├── ✅ Short Term Memory Memory Saved (209.04ms)
│   │   ├── ✅ Long Term Memory Memory Saved (1.81ms)
│   │   ├── ✅ Entity Memory Memory Saved (857.96ms)
│   │   ├── ✅ Short Term Memory Memory Saved (293.55ms)
│   │   ├── ✅ Long Term Memory Memory Saved (1.14ms)
│   │   └── ✅ Entity Memory Memory Saved (699.39ms)
│   └── ✅ Memory Retrieval Completed
│       └── Sources Used
│           ├── ✅ Long Term Memory (0.30ms)
│           ├── ✅ Short Term Memory (292.28ms)
│           └── ✅ Entity Memory (269.49ms)
├── ✅ Memory Retrieval Completed
                                      ...                                      

✅ Crew: crew
├── 📋 Task: Analyze Code Quality (ID: 952ee0e6-91ed-4f4c-9946-3e2750461c4e)
│   Assigned to: Senior Developer
│   
│   Status: ✅ Completed
├── 📋 Task: Review Security (ID: 417079a1-1992-46d3-9238-00e9c7d98508)
│   Assigned to: Security Engineer
│   
│   Status: ✅ Completed
│   ├── 🔧 Used Search the internet with Serper (1)
│   ├── 🧠 Memory Update Overall
│   │   ├── ✅ Short Term Memory Memory Saved (209.04ms)
│   │   ├── ✅ Long Term Memory Memory Saved (1.81ms)
│   │   ├── ✅ Entity Memory Memory Saved (857.96ms)
│   │   ├── ✅ Short Term Memory Memory Saved (293.55ms)
│   │   ├── ✅ Long Term Memory Memory Saved (1.14ms)
│   │   └── ✅ Entity Memory Memory Saved (699.39ms)
│   └── ✅ Memory Retrieval Completed
│       └── Sources Used
│           ├── ✅ Long Term Memory (0.30ms)
│           ├── ✅ Short Term Memory (292.28ms)
│           └── ✅ Entity Memory (269.49ms)
├── ✅ Memory Retrieval Completed
                                      ...                                      

✅ Crew: crew
├── 📋 Task: Analyze Code Quality (ID: 952ee0e6-91ed-4f4c-9946-3e2750461c4e)
│   Assigned to: Senior Developer
│   
│   Status: ✅ Completed
├── 📋 Task: Review Security (ID: 417079a1-1992-46d3-9238-00e9c7d98508)
│   Assigned to: Security Engineer
│   
│   Status: ✅ Completed
│   ├── 🔧 Used Search the internet with Serper (1)
│   ├── 🧠 Memory Update Overall
│   │   ├── ✅ Short Term Memory Memory Saved (209.04ms)
│   │   ├── ✅ Long Term Memory Memory Saved (1.81ms)
│   │   ├── ✅ Entity Memory Memory Saved (857.9

6ms)
│   │   ├── ✅ Short Term Memory Memory Saved (293.55ms)
│   │   ├── ✅ Long Term Memory Memory Saved (1.14ms)
│   │   └── ✅ Entity Memory Memory Saved (699.39ms)
│   └── ✅ Memory Retrieval Completed
│       └── Sources Used
│           ├── ✅ Long Term Memory (0.30ms)
│           ├── ✅ Short Term Memory (292.28ms)
│           └── ✅ Entity Memory (269.49ms)
├── ✅ Memory Retrieval Completed
✅ Crew: crew
├── 📋 Task: Analyze Code Quality (ID: 952ee0e6-91ed-4f4c-9946-3e2750461c4e)
│   Assigned to: Senior Developer
│   
│   Status: ✅ Completed
├── 📋 Task: Review Security (ID: 417079a1-1992-46d3-9238-00e9c7d98508)
│   Assigned to: Security Engineer
│   
│   Status: ✅ Completed
│   ├── 🔧 Used Search the internet with Serper (1)
│   ├── 🧠 Memory Update Overall
│   │   ├── ✅ Short Term Memory Memory Saved (209.04ms)
│   │   ├── ✅ Long Term Memory Memory Saved (1.81ms)
│   │   ├── ✅ Entity Memory Memory Saved (857.96ms)
│   │   ├── ✅ Short Term Memory Memory Saved (293.55ms)
│   │   ├── ✅ Long

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 5bb0e044-e562-4f5f-a1b6-3ed86fce45c2                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/5bb0e044-e562-4f5 │
│ f-a1b6-3ed86fce45c2?access_code=TRACE-dc267eaa94                             │
│ 🔑 Access Code: TRACE-dc267eaa94                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

**▶️ 在终端运行**

也可以在终端里跑；结束时 tracing 会给出 CrewAI 平台的追踪链接（链接被折行时拼起来再打开）。

```bash
cd code_review_flow
crewai run
```

</div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**`@persist` + 固定 id 的副作用**：每次运行先**恢复**上次存的 state，所以
- 改 `ReviewState` 里 `pr_file_path` 的**默认值不会生效**（恢复回来的旧值覆盖它）；
- `errors` 列表只追加不清空：某次读文件失败后，之后每次都会走 `"ERROR"` 分支。
想换文件或从头来：换一个 `id`，或在 `inputs` 里显式传 `pr_file_path`。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧪 在 notebook 里直接跑 SIMPLE 分支

</div>

不经过 CLI，在 kernel（crewai 1.15.22）里 import 同一个 Flow 类，用 **`await flow.kickoff_async(...)`**（Jupyter 已有事件循环，同步 `kickoff()` 内部的 `asyncio.run` 会报错）。
`inputs` 里的 key 会写进 state：这里换成简单的 diff，且**不传 id** → 全新 state，不会动上面 `my_code_review_flow` 的存档。kernel 的 cwd 是作业文件夹，所以路径不带 `../`。

In [13]:
sys.path.insert(0, str(SRC.parent))   # 让 `code_review_flow` 包可被 import
from code_review_flow.main import PRCodeReviewFlow

simple_flow = PRCodeReviewFlow()
answer = await simple_flow.kickoff_async(inputs={"pr_file_path": "files/code_changes_simple.txt"})

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: PRCodeReviewFlow                                                                                         │
│  ID: 42ce06c0-125a-4ce2-a654-3cd54afc3fd9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.22                                                                                       │
│  Latest version:  1.15.28                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: PRCodeReviewFlow                                                                                         │
│  ID: 42ce06c0-125a-4ce2-a654-3cd54afc3fd9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🔍 Starting PR Code Review...

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: read_pr_file                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: make_final_decision                                                                                    │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

📝 Final Answer:
- Final Review Decision: APPROVE
- Confidence Score: 90
- Findings: The pull request involves a modification to the import statement for the 'datetime' module, allowing it to be referenced using the alias 'dt'. This alters calls to 'datetime.now()' so that they now use 'dt.now()'. The changes are straightforward and improve code clarity by using an alias. 

Although the changes are largely a matter of preference, care should be taken to ensure that the alias 'dt' is used consistently throughout the file to avoid confusion. Additionally, a review of any dependencies on 'datetime' in other parts of the codebase is advisable to ensure no unintended side effects arise from this change.

- Recommendations:
  - Ensure consistent use of the alias 'dt' in all instances throughout the file.
  - Check for any potential impacts on other parts of the codebase that may reference 'datetime'. 

Overall, the PR shows quality enhancements and is acceptable for merging.

✨ Automatic code

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: return_final_answer                                                                                    │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: return_final_answer                                                                                    │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: PRCodeReviewFlow                                                                                         │
│  ID: 42ce06c0-125a-4ce2-a654-3cd54afc3fd9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [14]:
print("crew_needed =", simple_flow.state.crew_needed)   # 预期 False：只走了 simple_review
print("tokens_used =", simple_flow.state.tokens_used)   # 预期 {}：没跑 crew，就没有 token 统计

crew_needed = False
tokens_used = {}


<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 💰 可选：用 flow_state.json 算成本

</div>

`kickoff()` 把最后的 state 存进了 `flow_state.json`。读出 crew 的 token 数，按 gpt-4o-mini 价格（2025-09）计算：

In [15]:
import json
from pprint import pprint

# load the flow state from the JSON file
with open('flow_state.json', 'r') as f:
    flow_state = json.load(f)

# print the tokens_used variable
pprint(flow_state["tokens_used"])

# Pricing for gpt-4o-mini as of Sept 2025
input_cost_per_1M = 0.15   # 美元 / 百万 token
cached_input_cost_per_1M = 0.075
output_cost_per_1M = 0.6

prompt_tokens = flow_state["tokens_used"]["prompt_tokens"]
cached_tokens = flow_state["tokens_used"]["cached_prompt_tokens"]   # cached 是 prompt_tokens 的一部分
completion_tokens = flow_state["tokens_used"]["completion_tokens"]

input_cost = (prompt_tokens / 1_000_000) * input_cost_per_1M
cached_cost = (cached_tokens / 1_000_000) * cached_input_cost_per_1M
output_cost = (completion_tokens / 1_000_000) * output_cost_per_1M

total_cost = input_cost + output_cost + cached_cost

print("------------------------------------\nCost Breakdown:")
print(f"Input cost: ${input_cost:.6f}")
print(f"Output cost: ${output_cost:.6f}")
print(f"Cached cost: ${cached_cost:.6f}")
print(f"Total cost: ${total_cost:.6f}")

{'cached_prompt_tokens': 1024,
 'completion_tokens': 1368,
 'prompt_tokens': 7815,
 'successful_requests': 4,
 'total_tokens': 9183}
------------------------------------
Cost Breakdown:
Input cost: $0.001172
Output cost: $0.000821
Cached cost: $0.000077
Total cost: $0.002070


<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**课程公式重复计费了 cached tokens**：OpenAI 的 `cached_prompt_tokens` 是 `prompt_tokens` 的**子集**，上面把它们先按全价算一次、又按半价加一次。正确算法见下一格。
另外 `tokens_used` 只统计 **crew** 的用量；Flow 里直接 `LLM.call` 的 router 和最终决定那两次不在里面。

</div>

In [16]:
# 修正：缓存部分只按缓存价计一次
fixed_input_cost = ((prompt_tokens - cached_tokens) / 1_000_000) * input_cost_per_1M + cached_cost
print(f"Corrected total cost: ${fixed_input_cost + output_cost:.6f}  (course formula: ${total_cost:.6f})")

Corrected total cost: $0.001916  (course formula: $0.002070)


<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 坑**

| 坑 | 现象 | 原因 / 对策 |
|---|---|---|
| 项目 `.py` 是 notebook JSON | `SyntaxError` / import 失败 | 课程下载把文件转成了 ipynb 格式；要还原成纯 Python |
| `SerperDevTool(base_url=os.getenv(…))` 本地没设变量 | `base_url Input should be a valid string` | 给 `os.getenv` 加默认值 `https://google.serper.dev` |
| summarize 的 `context` 只写一个 | 可能在另一个 async 审查前开始 / 少了一半输入 | 两个上游都写进 context |
| 忘了 `@task` / `@agent` | yaml 按名字找不到 agent，或 task 没进 crew | 装饰器负责把方法收集进 `self.agents` / `self.tasks` |
| `tokens_used = result.token_usage` | state 序列化出问题 / json 里没有 key | `.model_dump()` 转成 dict |
| 改了默认 `pr_file_path` 没效果 | 还在审查旧文件 | `@persist` + 固定 id 恢复旧 state；换 id 或在 inputs 里传路径 |
| notebook 里 `flow.kickoff()` | `asyncio.run() cannot be called from a running event loop` | `await flow.kickoff_async(...)` |
| 成本按课程公式算 | 偏高 | cached 是 prompt 的子集，不能重复计费 |

</div>

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

**🔁 复习自测**（先自己答，再点开看答案）

<details><summary><b>1. 为什么 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">summarize_findings</code> 不设 async？</b></summary>

它需要两个审查的结果；同步 task 开始前 crew 会先等所有 async task 完成。而且 crew 末尾最多只能有一个 async task。

</details>

<details><summary><b>2. router 返回 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">&quot;ERROR&quot;</code> 时，哪些方法会运行？</b></summary>

只有 `return_final_answer`（它监听 `or_("ERROR", "make_final_decision")`）；两个审查和最终决定都不会触发。

</details>

<details><summary><b>3. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">kickoff(inputs={&#x27;file_content&#x27;: pr_content})</code> 的 key 为什么叫 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">file_content</code>？</b></summary>

tasks.yaml 的 description 里写的是 `{file_content}` 占位符，kickoff 时按名字替换。

</details>

<details><summary><b>4. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">result.json_dict</code> 来自哪个 task？</b></summary>

crew 最后一个 task `summarize_findings`，它设了 `output_json=SummarizedFindingsJSON`。

</details>

<details><summary><b>5. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">@persist</code> 存在哪里？固定 id 有什么副作用？</b></summary>

SQLite `flow_states.db`（用户数据目录下按项目名分文件夹）。同 id 再运行会恢复旧 state：默认值被覆盖、`errors` 会累积。

</details>

<details><summary><b>6. 这次 Flow 能省钱的关键在哪一步？</b></summary>

`analyze_changes`：一次便宜的 LLM 调用决定是否启动 crew，小改动不必跑三个 agent + 网络搜索。

</details>

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：**Flow 决定「要不要」和「走哪条路」，Crew 负责「怎么深挖」**：`@router` 用一次 LLM 判断分流，crew 内部用 async + context 并行审查再汇总，state 记下结果和 token 成本。

</div>